# llm-traffic-replay: smoke test (client correctness only)
Self-contained runnable payload (v0.6.0, 79 tracked files, 1111 pytest cases), unpacked to a fresh driver directory and run against a user-selected **pay-per-token** endpoint in this workspace at 0.1-1 QPS with the reduced validation profile. The cluster must provide Python 3.10+, NumPy 1.24+, and pytest 7+.

Set `extra_body_json` only to request controls documented for the exact model/provider contract (for example, an explicit reasoning control). These values are persisted as evidence; credential-like values are rejected.

**What this run proves:** auth path, streaming, TTFT-on-first-content capture, usage parsing, and which cached-token field this serving stack reports.

**What this run must never be quoted for: latency or performance.** Shared pay-per-token capacity does not establish dedicated-capacity performance. Dedicated benchmarking is a separate workload and is possible only for models that support that deployment mode.

In [ ]:
# Cell 1: unpack the embedded runnable payload to a fresh driver directory
import base64, hashlib, json, os, sys, tempfile
from pathlib import Path, PurePosixPath

PACKED_VERSION = "0.6.0"
EXPECTED_PAYLOAD_FILES = 79
PAYLOAD_SHA256 = "bac7f58b59b109f3dd73e136232f8b75c827981d10f311e3e1e768b6efecff09"
PAYLOAD = "..."

raw_payload = base64.b64decode(PAYLOAD, validate=True)
actual_digest = hashlib.sha256(raw_payload).hexdigest()
if actual_digest != PAYLOAD_SHA256:
    raise RuntimeError("embedded payload checksum mismatch; stop")
# Bootstrap exception: this digest-authenticated payload contains the strict parser itself.
payload_files = json.loads(raw_payload)
if not isinstance(payload_files, dict) or len(payload_files) != EXPECTED_PAYLOAD_FILES:
    raise RuntimeError("embedded payload file count mismatch; stop")
root = Path(tempfile.mkdtemp(prefix="llm-traffic-replay-"))
for rel, text in payload_files.items():
    pure = PurePosixPath(rel)
    if pure.is_absolute() or not pure.parts or ".." in pure.parts or not isinstance(text, str):
        raise RuntimeError(f"unsafe embedded payload entry: {rel!r}")
    p = root.joinpath(*pure.parts)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_bytes(text.encode("utf-8"))
preloaded = sorted(name for name in sys.modules if name == "traffic_replay" or name.startswith("traffic_replay."))
if preloaded:
    raise RuntimeError(f"traffic_replay was already imported ({preloaded[:3]}); restart Python and rerun from Cell 1")
os.chdir(root)
sys.path.insert(0, str(root))
import traffic_replay
if (traffic_replay.__version__ != PACKED_VERSION
        or Path(traffic_replay.__file__).resolve().parent != (root / "traffic_replay").resolve()):
    raise RuntimeError("embedded package version or import origin mismatch; stop")
print("unpacked exact payload to", root, "|", len(payload_files), "files | sha256", actual_digest)

In [ ]:
# Cell 2: run the full pytest suite (1111 cases) + instrument validation
import json, os, re, subprocess, sys, tempfile
import xml.etree.ElementTree as ET
from importlib.metadata import PackageNotFoundError, version as distribution_version
from pathlib import Path
EXPECTED_PYTEST_CASES = 1111
try:
    pytest_major = int(distribution_version("pytest").split(".", 1)[0])
except (PackageNotFoundError, ValueError, TypeError) as exc:
    raise RuntimeError("cannot determine the installed pytest version; stop") from exc
if pytest_major < 7:
    raise RuntimeError("pytest 7 or newer is required; stop")
try:
    numpy_match = re.match(r"^(\d+)\.(\d+)", distribution_version("numpy"))
except PackageNotFoundError as exc:
    raise RuntimeError("NumPy 1.24 or newer is required; stop") from exc
if not numpy_match or tuple(map(int, numpy_match.groups())) < (1, 24):
    raise RuntimeError("NumPy 1.24 or newer is required; stop")
pytest_env = os.environ.copy()
pytest_env["PYTEST_DISABLE_PLUGIN_AUTOLOAD"] = "1"
pytest_env.pop("PYTEST_ADDOPTS", None)
pytest_env.pop("PYTEST_PLUGINS", None)
pytest_env.pop("PYTHONPATH", None)

def run_checked(command, timeout_s, label, env=None):
    try:
        result = subprocess.run(command, capture_output=True, text=True, timeout=timeout_s, env=env)
    except subprocess.TimeoutExpired as exc:
        raise RuntimeError(f"{label} exceeded {timeout_s} seconds; stop") from exc
    if result.returncode != 0:
        print(result.stdout[-3000:])
        print(result.stderr[-2000:], file=sys.stderr)
        raise RuntimeError(f"{label} failed with exit code {result.returncode}; stop")
    return result

collect = run_checked([sys.executable, "-m", "pytest", "--collect-only", "-q", "-o", "addopts=", "-p", "no:cacheprovider"], 180, "pytest collection", pytest_env)
nodeids = [line for line in collect.stdout.splitlines() if line.startswith("tests/") and "::" in line]
collection_summary = re.search(r"(?m)^(\d+) tests? collected\b", collect.stdout)
if (not collection_summary or int(collection_summary.group(1)) != len(nodeids)
        or len(nodeids) != len(set(nodeids)) or len(nodeids) != EXPECTED_PYTEST_CASES):
    raise RuntimeError(f"pytest collection mismatch: expected {EXPECTED_PYTEST_CASES}, got {len(nodeids)}; stop")
junit_path = Path(tempfile.mkdtemp(prefix="llm-traffic-replay-pytest-")) / "results.xml"
tests = run_checked([sys.executable, "-m", "pytest", "-q", "-o", "addopts=", "-p", "no:cacheprovider", f"--junitxml={junit_path}"], 1200, "pytest suite", pytest_env)
suites = ET.parse(junit_path).getroot().findall("testsuite")
if len(suites) != 1:
    raise RuntimeError(f"pytest JUnit evidence has {len(suites)} suites, expected one; stop")
junit_counts = {name: int(suites[0].attrib.get(name, "0")) for name in ("tests", "failures", "errors", "skipped")}
if (junit_counts["tests"] != EXPECTED_PYTEST_CASES
        or any(junit_counts[name] != 0 for name in ("failures", "errors", "skipped"))):
    raise RuntimeError(f"pytest JUnit counts disagree with collection: {junit_counts}; stop")
print(tests.stdout[-1200:])
validation_root = tempfile.mkdtemp(prefix="llm-traffic-replay-validation-")
validation_run = run_checked([sys.executable, "-m", "traffic_replay", "validate", "--quiet", "--format", "json", "--port", "0", "--duration", "15", "--tolerance-ms", "120", "--workdir", validation_root], 240, "instrument validation", pytest_env)
from traffic_replay.json_input import loads_strict
validation = loads_strict(validation_run.stdout)
if validation.get("passed") is not True or validation.get("joined_requests", 0) < 1:
    raise RuntimeError("instrument validation did not produce passing joined evidence; stop")
print(json.dumps(validation, indent=2, allow_nan=False))

In [ ]:
# Cell 3: require an explicit endpoint and durable Volume destination
import urllib.error, urllib.parse, urllib.request, uuid
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

dbutils.widgets.text("endpoint_name", "", "Pay-per-token endpoint name (required)")
dbutils.widgets.text("artifact_volume_path", "", "Durable /Volumes/... artifact root (required)")
dbutils.widgets.text("extra_body_json", "{}", "Provider-documented request controls (JSON object)")
dbutils.widgets.dropdown("confirm_paid_smoke", "NO", ["NO", "RUN"], "Confirm 60-second paid smoke")
ENDPOINT = dbutils.widgets.get("endpoint_name").strip()
artifact_value = dbutils.widgets.get("artifact_volume_path").strip()
from traffic_replay.json_input import loads_strict
try:
    EXTRA_BODY = loads_strict(dbutils.widgets.get("extra_body_json"))
except ValueError as exc:
    raise ValueError("extra_body_json must be valid JSON") from exc
from traffic_replay.client import validate_extra_body_safety
validate_extra_body_safety(EXTRA_BODY)
if dbutils.widgets.get("confirm_paid_smoke") != "RUN":
    raise ValueError("set confirm_paid_smoke to RUN after reviewing the 0.1-1 QPS paid workload")
if not ENDPOINT:
    raise ValueError("set the endpoint_name widget explicitly; no endpoint is auto-selected")
if (ENDPOINT in (".", "..") or "/" in ENDPOINT or "\\" in ENDPOINT
        or any(ord(char) < 32 or ord(char) == 127 for char in ENDPOINT)):
    raise ValueError("endpoint_name contains unsafe path characters")
ENDPOINT_PATH_NAME = urllib.parse.quote(ENDPOINT, safe="-_.~")
artifact_path = PurePosixPath(artifact_value)
if (not artifact_value.startswith("/Volumes/") or len(artifact_path.parts) < 5
        or artifact_path.parts[1] != "Volumes" or ".." in artifact_path.parts):
    raise ValueError("artifact_volume_path must be /Volumes/<catalog>/<schema>/<volume>[/subdir]")
ARTIFACT_ROOT = Path(artifact_value)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
SMOKE_OUTPUT_BASE = ARTIFACT_ROOT / f"smoke-{stamp}-{uuid.uuid4().hex[:12]}"
if SMOKE_OUTPUT_BASE.exists():
    raise RuntimeError(f"fresh artifact base already exists: {SMOKE_OUTPUT_BASE}")

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
browser_host = str(ctx.browserHostName().get()).strip()
TOKEN = str(ctx.apiToken().get()).strip()
if not browser_host or "/" in browser_host or not TOKEN:
    raise RuntimeError("workspace host or ambient notebook token is unavailable")
HOST = "https://" + browser_host
endpoint_url = HOST + "/api/2.0/serving-endpoints/" + ENDPOINT_PATH_NAME
request = urllib.request.Request(endpoint_url, headers={"Authorization": f"Bearer {TOKEN}"})
try:
    with urllib.request.urlopen(request, timeout=15) as response:
        discovery_bytes = response.read(4 * 1024 * 1024 + 1)
except (urllib.error.HTTPError, urllib.error.URLError, TimeoutError) as exc:
    raise RuntimeError(f"bounded endpoint discovery failed: {type(exc).__name__}") from exc
if len(discovery_bytes) > 4 * 1024 * 1024:
    raise RuntimeError("endpoint discovery response exceeded 4 MiB")
discovery = loads_strict(discovery_bytes)
if not isinstance(discovery, dict) or not discovery:
    raise RuntimeError("endpoint discovery returned an empty or invalid object")
if discovery.get("name") != ENDPOINT:
    raise ValueError(f"endpoint discovery did not return the explicit target {ENDPOINT!r}")
selected = discovery
endpoint_type = selected.get("endpoint_type")
if endpoint_type is not None and endpoint_type != "FOUNDATION_MODEL_API":
    raise ValueError(f"explicit endpoint is not advertised as pay-per-token: {endpoint_type!r}")
if endpoint_type is None:
    print("endpoint metadata does not establish deployment mode; explicit pay-per-token selection remains required")
task = selected.get("task")
if task is not None and task not in ("llm/v1/chat", "chat/completions"):
    raise ValueError(f"explicit endpoint does not advertise a chat task: {task!r}")
state = selected.get("state")
ready = state.get("ready") if isinstance(state, dict) else None
if ready != "READY":
    raise RuntimeError(f"explicit endpoint is not READY: {ready!r}")
print("explicit endpoint verified:", ENDPOINT)
print("provider request-control keys:", sorted(EXTRA_BODY))
print("durable artifact base:", SMOKE_OUTPUT_BASE)

In [ ]:
# Cell 4: 60-second correctness smoke at 0.1-1 QPS with capped outputs
from pathlib import Path
from traffic_replay.runner import RunConfig, run

rc = RunConfig(
    profile_path="configs/profile_validation_small.json",
    endpoint={"base_url": HOST,
              "path": f"/serving-endpoints/{ENDPOINT_PATH_NAME}/invocations",
              "auth_token_env": "UNUSED", "extra_body": EXTRA_BODY,
              "total_timeout_s": 60.0},
    duration_s=60, qps_base=0.25, qps_burst=0.75, qps_min=0.1, qps_max=1.0,
    max_concurrency=8, cpt=4.0, calibrate_n=4, seed=7,
    out_dir=str(SMOKE_OUTPUT_BASE), title=f"smoke vs {ENDPOINT} (client correctness only)",
    label="SMOKE TEST on shared pay-per-token capacity: NOT performance evidence.",
    max_output_tokens_cap=24)
out = run(rc, token_override=TOKEN)
run_dir = Path(out["out_dir"])
if run_dir.parent != SMOKE_OUTPUT_BASE or not run_dir.is_dir():
    raise RuntimeError("runner returned an unexpected artifact directory; stop")
print("sealed run awaits integrity and correctness verification:", run_dir)

In [ ]:
# Cell 5: verify sealed artifacts and correctness gates, then print the report verbatim
import hashlib
from pathlib import Path
from traffic_replay.json_input import loads_strict
run_dir = Path(out["out_dir"])
required = ["start.json", "requests.jsonl", "summary.json", "report.md", "report.html", "manifest.json", ".traffic-replay-complete"]
missing = [name for name in required if not (run_dir / name).is_file()]
if missing:
    raise RuntimeError(f"artifact set is incomplete, missing: {missing}")
manifest_raw = (run_dir / "manifest.json").read_bytes()
manifest = loads_strict(manifest_raw)
complete = loads_strict((run_dir / ".traffic-replay-complete").read_bytes())
manifest_digest = hashlib.sha256(manifest_raw).hexdigest()
artifact_id = manifest.get("artifact_id")
if (manifest.get("manifest_schema_version") != 3 or complete.get("status") != "complete"
        or complete.get("manifest_sha256") != manifest_digest
        or complete.get("manifest_bytes") != len(manifest_raw)
        or not isinstance(artifact_id, str) or not artifact_id
        or complete.get("artifact_id") != artifact_id):
    raise RuntimeError("completion marker does not bind the v3 manifest; stop")
artifacts = manifest.get("artifacts")
if not isinstance(artifacts, dict) or not artifacts:
    raise RuntimeError("manifest has no artifact bindings; stop")
required_bound = {"start.json", "requests.jsonl", "summary.json", "report.md", "report.html"}
if not required_bound.issubset(artifacts):
    raise RuntimeError(f"manifest omits required artifact bindings: {sorted(required_bound - artifacts.keys())}")
for name, expected in artifacts.items():
    if Path(name).name != name or not isinstance(expected, dict):
        raise RuntimeError(f"unsafe manifest artifact entry: {name!r}")
    raw = (run_dir / name).read_bytes()
    if expected.get("bytes") != len(raw) or expected.get("sha256") != hashlib.sha256(raw).hexdigest():
        raise RuntimeError(f"artifact integrity check failed: {name}")
requests_expected = artifacts.get("requests.jsonl", {}).get("row_count")
requests_raw = (run_dir / "requests.jsonl").read_bytes()
if requests_raw and not requests_raw.endswith(b"\n"):
    raise RuntimeError("requests.jsonl has an incomplete final record")
request_lines = requests_raw.splitlines()
if any(not line.strip() for line in request_lines):
    raise RuntimeError("requests.jsonl contains an empty record")
replay_rows = 0
for line_number, line in enumerate(request_lines, 1):
    try:
        request_row = loads_strict(line)
    except ValueError as exc:
        raise RuntimeError(f"requests.jsonl row {line_number} is invalid JSON") from exc
    if not isinstance(request_row, dict):
        raise RuntimeError(f"requests.jsonl row {line_number} is not an object")
    replay_rows += request_row.get("phase") == "replay"
if (not isinstance(requests_expected, int) or isinstance(requests_expected, bool)
        or requests_expected < 0 or requests_expected != len(request_lines)
        or requests_expected != complete.get("request_rows")):
    raise RuntimeError("request row count disagrees across manifest and completion marker")
summary = loads_strict((run_dir / "summary.json").read_bytes())
if not isinstance(summary, dict):
    raise RuntimeError("sealed summary is not an object; stop")
requests_total = summary.get("requests_total")
answers = summary.get("answers")
ttft = summary.get("ttft_ms")
usage_coverage = (summary.get("throughput") or {}).get("usage_coverage")
cache = summary.get("achieved_cache_fraction") or {}
if (not isinstance(requests_total, int) or isinstance(requests_total, bool) or requests_total < 1
        or replay_rows != requests_total or summary.get("requests_ok") != requests_total
        or summary.get("requests_failed") != 0):
    raise RuntimeError("correctness smoke had missing or failed replay requests; sealed evidence is not green")
if (not isinstance(answers, dict) or answers.get("judged") != requests_total
        or answers.get("answered") != requests_total or answers.get("answer_rate") != 1.0
        or answers.get("stream_incomplete") != 0 or answers.get("parse_errors") != 0):
    raise RuntimeError("not every replay request produced one clean readable answer; sealed evidence is not green")
if not isinstance(ttft, dict) or ttft.get("n") != requests_total or ttft.get("missing") != 0:
    raise RuntimeError("first-content TTFT was not measured for every replay request; sealed evidence is not green")
if usage_coverage != 1.0:
    raise RuntimeError("endpoint token-usage coverage was not 100%; sealed evidence is not green")
cache_sources = cache.get("source_fields")
if (cache.get("reported_for_n") != requests_total or cache.get("coverage") != 1.0
        or not isinstance(cache_sources, list) or not cache_sources
        or any(source in ("NOT REPORTED BY ENDPOINT", "SOURCE FIELD NOT RECORDED") for source in cache_sources)):
    raise RuntimeError("cached-token usage and its source field were not reported for every replay request")
print((run_dir / "report.md").read_text(encoding="utf-8"))
print("verified sealed durable evidence:", run_dir, "| manifest sha256", manifest_digest)